# 第 4 章 · 一个完整的 CUDA 程序：分配、拷贝、启动、同步、查错

本章回答：kernel 写好了，CPU 端还要做哪些事才能让它跑起来？数据怎么进出显存？为什么 kernel 启动后 CPU 不等它就往下走？出错了去哪里看？

配套讲义：仓库里 `cuda-programming/notes/lessons/04-host-device-program.md`（讲解、动画、练习都在讲义里）。对应原文：PG §2.1.3.2 Explicit Memory Management（`materials/text/cuda-programming-guide.txt` 第 1877–1994 行）、§2.1.4 Synchronizing CPU and GPU（第 1995–2008 行）、§2.1.5 Putting it All Together（第 2009–2368 行）、§2.1.7 Error Checking（第 2369–2470 行）；Best Practices §10.1 Data Transfer（`text/cuda-c-best-practices-guide.txt` 第 1085–1336 行）；Oxford lec1 第 23–30 页、prac1

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU：代码用 NumPy 在 CPU 上模拟 GPU 的规则（线程编号、32 字节一箱、32 个 bank……）。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 五步走 | 分配、拷进、启动、同步、拷回 | `gpu.malloc` → `memcpy_h2d` → `launch` → `memcpy_d2h` | C = [1, 2, …, 8] |
| 第三步 启动是下单 | 异步启动 | `launch` 只把 kernel 放进 `queue` | 刚启动完 C 全是 0 |
| 第四步 错误会晚到 | `cudaGetLastError` | `launch(bad_kernel)` 返回 `cudaSuccess`，`synchronize()` 才报错 | `cudaErrorIllegalAddress` |
| 第五步 搬运账 | 字节 ÷ 带宽 | `transfer_ms(800, 64)` 等 | 12.50、0.36、6.25 ms，合计 ≈ 19.1 ms |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**类 `FakeGPU`**（来自 `ch04_host_device.py`）：一块"显存" + 一个任务队列。kernel 启动只是入队，同步时才真正执行。

In [ ]:
class FakeGPU:
    """一块"显存" + 一个任务队列。kernel 启动只是入队，同步时才真正执行。"""

    def __init__(self):
        self.mem: dict[str, np.ndarray] = {}   # device 内存：名字 → 数组（CPU 不能直接下标访问）
        self.queue: list = []                  # 已下单、还没做的 kernel
        self.pending_error: str | None = None  # kernel 运行中出的错，要到同步点才报告

    def malloc(self, name: str, n: int):                       # ① cudaMalloc
        self.mem[name] = np.zeros(n, dtype=np.float32)

    def memcpy_h2d(self, name: str, host: np.ndarray):         # ② H2D
        self.synchronize()
        self.mem[name][:] = host

    def launch(self, kernel, *args) -> str:                    # ③ kernel<<<...>>>：只入队，立刻返回
        self.queue.append((kernel, args))
        return "cudaSuccess"                                   # 启动参数没问题 ≠ kernel 跑成功

    def synchronize(self) -> str:                              # ④ cudaDeviceSynchronize
        for kernel, args in self.queue:
            try:
                kernel(self.mem, *args)
            except IndexError:
                self.pending_error = "cudaErrorIllegalAddress"
        self.queue.clear()
        err, self.pending_error = self.pending_error or "cudaSuccess", None
        return err

    def memcpy_d2h(self, name: str) -> tuple[str, np.ndarray]:  # ④ D2H：先等前面的 kernel 做完
        err = self.synchronize()
        return err, self.mem[name].copy()

**函数 `bad_kernel`**（来自 `ch04_host_device.py`）：越界写：写到 n + 5 的位置。

In [ ]:
def bad_kernel(mem, a: str, n: int):
    """越界写：写到 n + 5 的位置。"""
    mem[a][n + 5] = 1.0

**函数 `transfer_ms`**（来自 `ch04_host_device.py`）：搬运时间（毫秒）= 字节数 ÷ 带宽。

In [ ]:
def transfer_ms(megabytes: float, gb_per_s: float) -> float:
    """搬运时间（毫秒）= 字节数 ÷ 带宽。"""
    return megabytes * 1e6 / (gb_per_s * 1e9) * 1e3

**函数 `vec_add_kernel`**（来自 `ch04_host_device.py`）：下面要用到的小工具。

In [ ]:
def vec_add_kernel(mem, a: str, b: str, c: str, n: int):
    mem[c][:n] = mem[a][:n] + mem[b][:n]

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二、三步 五步走 + 异步启动")
n = 8
gpu = FakeGPU()
for name in ("A", "B", "C"):
    gpu.malloc(name, n)
gpu.memcpy_h2d("A", np.arange(n, dtype=np.float32))
gpu.memcpy_h2d("B", np.ones(n, dtype=np.float32))
print("  启动返回：", gpu.launch(vec_add_kernel, "A", "B", "C", n))
print("  刚启动完，C 里还是：", gpu.mem["C"], "（kernel 只是排进了队列）")
err, C = gpu.memcpy_d2h("C")
print("  拷回以后 C =", C, "，状态", err)

In [ ]:
print("\n第四步 错误会晚到")
gpu2 = FakeGPU()
gpu2.malloc("A", n)
print("  启动越界 kernel，cudaGetLastError →", gpu2.launch(bad_kernel, "A", n))
print("  到了同步点才冒出来 →", gpu2.synchronize())

In [ ]:
print("\n第五步 n = 1 亿个 float，每个数组 400 MB")
rows = [("H2D 拷 A、B", 800, 64), ("kernel 读 A、B 写 C", 1200, 3350), ("D2H 拷 C", 400, 64)]
total = 0.0
for what, mb, bw in rows:
    ms = transfer_ms(mb, bw)
    total += ms
    print(f"  {what:<12} {mb:5d} MB ÷ {bw:5d} GB/s ≈ {ms:6.2f} ms")
print(f"  合计 ≈ {total:.1f} ms，其中 kernel 只占 {transfer_ms(1200, 3350) / total:.1%}")

## 3. 代码里才看得到的

1. **`FakeGPU.mem` 是一个字典，CPU 端只能通过 `memcpy_*` 拿数据**：对应讲义说的"`devA` 是显存地址，CPU 不能直接 `devA[0]`"。
2. **`memcpy_d2h` 里先调了 `synchronize()`**：这就是"不带 Async 的 `cudaMemcpy` 会等前面的 kernel 做完"。
3. **D2H 算出来是 6.25 ms**：讲义表里写 ≈ 6.3 ms 是四舍五入到一位小数，测试按 6.25 核对。

## 4. 自己改一改

- 把第五步的 PCIe 带宽从 64 GB/s 改成 NVLink-C2C 的量级（比如 450），看搬运和 kernel 的比例怎么变。
- 在 `bad_kernel` 后面再 `launch` 一个正常的 `vec_add_kernel`，看错误是在哪一次 `synchronize()` 冒出来的。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 两个仓库一条窄路：能不搬就不搬，搬了就多干几道；
> 五步走：分配、拷进、启动、同步、拷回（最后释放）；
> 启动是下单不是等饭，要结果先同步，计时也先同步；
> 尖括号不报错，启动后查 `cudaGetLastError`；kernel 里的错会晚到，查不清就上 compute-sanitizer。